## Main Scripts for Producing All Virtual Staining Results

1. AF -> HE
    * UTOM CycleGAN
    * UTOM CycleGAN for vH (nuclei only), trained with unpaired TPAF nuclei masks and H&E Hematoxylin channel images, the vH output needs to be multiplied with TPAF nuclei masks to remove non-nuclei content
2. HE -> AF

In [ ]:
# Import necessary libraries

import os

### AF -> HE

Current best performing AF -> HE virtual staining models:
1. For 1 channel gray AF -> 3 channel HE: set37_train_1channel_saliency_A65_B220
2. For 3 channel RGB AF -> 3 channel HE: set37_train_set37_unpaired2500+paired2500_512_A70_B220_lambda15

In [ ]:
# 1 channel gray AF -> 3 channel HE
dataroot = './datasets/test_data/250711_slides'
epoch = 80
direction = 'AtoB'
results_dir = './datasets/test_data/250711_slides/04_results_TPAF_gray_patches_nuc_replaced_with_vH'
checkpoint_name = 'set37_train_1channel_saliency_A65_B220'
input_nc = 1
output_nc = 3
gpu_ids = 0
os.system(f'python test.py --dataroot {dataroot} --epoch {epoch} --direction {direction} \
           --results_dir {results_dir} --name {checkpoint_name} --model cycle_gan \
           --trainA_normalize 255 --trainB_normalize 255 --input_nc {input_nc} --output_nc {output_nc} \
           --gpu_ids {gpu_ids} --load_size 512 --crop_size 512 --display_winsize 512 --num_test 999')

In [ ]:
# 3 channel RGB AF -> 3 channel HE
dataroot = './datasets/test_data/250711_slides'
epoch = 100
direction = 'AtoB'
results_dir = './datasets/test_data/250711_slides/05_results_vHE_patches_nuc_hi'
checkpoint_name = 'set37_train_set37_unpaired2500+paired2500_512_A70_B220_lambda15'
input_nc = 3
output_nc = 3
gpu_ids = 0
os.system(f'python test.py --dataroot {dataroot} --epoch {epoch} --direction {direction} \
           --results_dir {results_dir} --name {checkpoint_name} --model cycle_gan \
           --trainA_normalize 255 --trainB_normalize 255 --input_nc {input_nc} --output_nc {output_nc} \
           --gpu_ids {gpu_ids} --load_size 512 --crop_size 512 --display_winsize 512 --num_test 999')

### Combine TPAF grayscale patches with vH output (non-nuclei content removed)

In [ ]:
import os
import cv2
import numpy as np
import tifffile
from tqdm import tqdm

from scipy.ndimage import binary_opening

In [ ]:
def replace_pixels(tpaf, nuc, intensity_thresh=25, approx_epsilon=2.0):
    """
    Replace TPAF pixels with nuclei signal where nuc > intensity_thresh,
    and smooth the boundaries using contour smoothing.
    
    Args:
        tpaf: 2D np.ndarray (grayscale)
        nuc: 2D np.ndarray (grayscale)
        intensity_thresh: minimum value in nuc to be considered valid signal
        approx_epsilon: smoothing factor for contour simplification
    Returns:
        Modified TPAF image (same dtype as input)
    """
    tpaf = tpaf.astype(np.float32)
    nuc = nuc.astype(np.float32)
    
    # Step 1: Binary mask where nuc > threshold
    binary_mask = (nuc > intensity_thresh).astype(np.uint8) * 255

    # Step 2: Find contours
    contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Step 3: Smooth contours with approxPolyDP
    smooth_mask = np.zeros_like(binary_mask)
    for cnt in contours:
        epsilon = approx_epsilon * cv2.arcLength(cnt, True) / 100.0
        smoothed = cv2.approxPolyDP(cnt, epsilon, True)
        cv2.drawContours(smooth_mask, [smoothed], -1, color=255, thickness=-1)

    # Step 4: Create mask and apply replacement
    mask = smooth_mask.astype(bool)
    result = tpaf.copy()
    result[mask] = nuc[mask]
    return result.astype(tpaf.dtype)

def process_dirs(tpaf_dir, nuc_dir, out_dir):
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    tpaf_files = sorted([f for f in os.listdir(tpaf_dir) if f.lower().endswith(('.tif','.tiff', '.png', '.jpg'))])
    nuc_files = sorted([f for f in os.listdir(nuc_dir) if f.lower().endswith(('.tif','.tiff', '.png', '.jpg'))])
    assert len(tpaf_files) == len(nuc_files), "两目录图像数量不一致"
    
    for fn_tpaf, fn_nuc in tqdm(zip(tpaf_files, nuc_files), total=len(tpaf_files)):
        if fn_tpaf.endswith('.tif') or fn_tpaf.endswith('.tiff'):
            tpaf = tifffile.imread(os.path.join(tpaf_dir, fn_tpaf))
            nuc = tifffile.imread(os.path.join(nuc_dir, fn_nuc))
        elif fn_tpaf.endswith('.png') or fn_tpaf.endswith('.jpg'):
            tpaf = cv2.imread(os.path.join(tpaf_dir, fn_tpaf), cv2.IMREAD_UNCHANGED)
            nuc = cv2.imread(os.path.join(nuc_dir, fn_nuc), cv2.IMREAD_UNCHANGED)

        if tpaf.shape != nuc.shape:
            raise ValueError(f"Shape mismatch for {fn_tpaf} vs {fn_nuc}: {tpaf.shape} vs {nuc.shape}")

        out = replace_pixels(tpaf, nuc)
        #tifffile.imwrite(os.path.join(out_dir, fn_tpaf), out, dtype=out.dtype)
        cv2.imwrite(os.path.join(out_dir, fn_tpaf), out)
        
    print("Finished processing all images.")

In [ ]:
root_dir = "C:/Users/zpanp/projects/UTOM-master/datasets/test_data/250711_slides"
TPAF_dir = os.path.join(root_dir, "02_og_TPAF_gray_patches")
nuc_dir = os.path.join(root_dir, "06_results_vH_patches_background_remove")
out_dir = os.path.join(root_dir, "03_TPAF_gray_patches_nuc_replaced_with_vH")

process_dirs(TPAF_dir, nuc_dir, out_dir)

# 

### HE -> AF

Current best performing HE -> virtual staining models:

1. For 3 channel HE -> 3 channel RGB AF: set37_train_set37_unpaired2500+paired2500_512_A70_B220_lambda15

In [ ]:
# 3 channel HE -> 3 channel RGB AF
dataroot = ''
epoch = 100
direction = 'BtoA'
results_dir = ''
checkpoint_name = 'set37_train_set37_unpaired2500+paired2500_512_A70_B220_lambda15'
input_nc = 3
output_nc = 3
gpu_ids = 0
os.system(f'python test.py --dataroot {dataroot} --epoch {epoch} --direction {direction} \
           --results_dir {results_dir} --name {checkpoint_name} --model cycle_gan \
           --trainA_normalize 255 --trainB_normalize 255 --input_nc {input_nc} --output_nc {output_nc} \
           --gpu_ids {gpu_ids} --load_size 512 --crop_size 512 --display_winsize 512 --num_test 999')